# Model Training (Keras)

## Library

In [210]:
import os
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_NVML_BASED_CUDA_CHECK'] = '1'
os.environ['KERAS_BACKEND'] = 'torch'
import torch

print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
  print('Device Name:', torch.cuda.get_device_name(0))
import keras
from keras.models import Sequential, load_model
from keras.layers import Input, Dense, Activation, Dropout, LeakyReLU
from keras.regularizers import l2
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras_tuner import HyperModel, RandomSearch

from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
import numpy as np
import pandas as pd
import random
from parameters import *
from pathLib import Path
home = Path.home()

GPU Available: True
Device Name: NVIDIA A100 80GB PCIe MIG 4g.40gb


ModuleNotFoundError: No module named 'pathLib'

In [211]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Dense, Activation, Dropout, LeakyReLU
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import os
import numpy as np
import pandas as pd

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from keras_tuner import HyperModel, RandomSearch
from tensorflow.keras.models import load_model

# Disable some console warnings
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'


#validating GPUs with tf
from tensorflow.python.client import device_lib
print("GPUs Available:", tf.config.list_physical_devices('GPU'))

GPUs Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [212]:
seed = 0
random.seed(seed)
np.random.seed(seed)

# PyTorch (covers CPU and all GPU devices like your A100)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Optional: ensure deterministic operations for strict reproducibility (may slow down training slightly)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Dataset

### Load

In [213]:
# Keeping DATA_AUGMENTATION 'True' for now so everything scales between 0 and 1
if DATA_AUGMENTATION:
    y_train_val = np.load('{}/npy/y_{}_train_val_augmented.npy'.format(DATA_DIR, PARAM), allow_pickle=True)
    y_test = np.load('{}/npy/y_{}_test_augmented.npy'.format(DATA_DIR, PARAM), allow_pickle=True) 

    X_train_val = np.load('{}/npy/x_{}_train_val_augmented.npy'.format(DATA_DIR, PARAM), allow_pickle=True)
    X_test = np.load('{}/npy/x_{}_test_augmented.npy'.format(DATA_DIR, PARAM), allow_pickle=True)
    
    X_train_val_log = np.load('{}/npy/x_{}_train_val_augmented_log.npy'.format(DATA_DIR, PARAM), allow_pickle=True)
    X_test_log = np.load('{}/npy/x_{}_test_augmented_log.npy'.format(DATA_DIR, PARAM), allow_pickle=True)
else:
    y_train_val = np.load('{}/npy/y_{}_train_val.npy'.format(DATA_DIR, PARAM), allow_pickle=True)
    y_test = np.load('{}/npy/y_{}_test.npy'.format(DATA_DIR, PARAM), allow_pickle=True)

    X_train_val = np.load('{}/npy/x_{}_train_val.npy'.format(DATA_DIR, PARAM), allow_pickle=True)
    X_test = np.load('{}/npy/x_{}_test.npy'.format(DATA_DIR, PARAM), allow_pickle=True)
    
    X_train_val_log = np.load('{}/npy/x_{}_train_val_log.npy'.format(DATA_DIR, PARAM), allow_pickle=True)
    X_test_log = np.load('{}/npy/x_{}_test_log.npy'.format(DATA_DIR, PARAM), allow_pickle=True)

### Visualize

In [214]:
def getSize(array):
    processed_lists = [
        [col[1:] if isinstance(col, (list, np.ndarray)) else col for col in sample] 
        for sample in array
    ]
    
    # 2. Pre-allocate a 2D NumPy array of objects to keep dimensions strictly (18846, 3)
    array = np.empty((len(processed_lists), 3), dtype=object)
    
    # 3. Populate it so each of the 3 columns remains a native Python list
    for i, sample in enumerate(processed_lists):
        array[i, :] = sample
    
    # 4. Verify shape and types
    print("Shape:", array.shape)              # Will be (18846, 3)
    print("Type of column 0:", type(array[0, 0])) # Will be <class 'list'>
    print("Length of that list:", len(array[0, 0]))
    return array

In [215]:
print(y_train_val.shape)
print(X_train_val.shape[1])

(4312, 7)
3


In [216]:
if len(X_train_val[1][2]) != 185:
    X_train_val = getSize(X_train_val)
if len(X_train_val_log[1][2]) != 185:
    X_train_val_log = getSize(X_train_val_log)
if len(X_test[1][2]) != 185:
    X_test = getSize(X_test)
if len(X_test_log[1][2]) != 185:
    X_test_log = getSize(X_test_log)
    


print(len(X_test_log[0][0]))
print(len(X_test[0][0]))
print(len(X_train_val[1][2]))
print(len(X_train_val_log[1][2]))

#print(X_train_val[0][2])
print(y_test[0])

185
185
185
185
[0.20913952 0.05869734 0.93277418 0.90966688 0.30719504 0.49961605
 0.76588046]


In [148]:
#print(X_train_val[0][0])
#X_train_val = X_train_val[..., 1:]
print(len(X_train_val[0][0]))
print(len(X_train_val_log[0][0]))

185
185


In [149]:
#number of values in the training set and test set, these are probably pulling from the data files individual points

total = len(y_train_val) + len(y_test)
print('---------------------------------------')  
print('Train-and-validation set shape: {}, {:.2f}%'.format(len(y_train_val), (len(y_train_val)*100.)/total))
print('Test set shape:                 {}, {:.2f}%'.format(len(y_test), (len(y_test)*100.)/total))
print('---------------------------------------')

---------------------------------------
Train-and-validation set shape: 4312, 79.99%
Test set shape:                 1079, 20.01%
---------------------------------------


In [150]:
#%matplotlib ipympl
%matplotlib inline
import matplotlib.pyplot as plt
#from mpl_toolkits.mplot3d import Axes3D

In [151]:
'''fig = plt.figure(figsize=(12, 12))
ax = fig.add_subplot(projection='3d')

for i in range(len(X_train_val)):
    Vg, Vd, Id = X_train_val[i]
    ax.scatter(Vg, Vd, Id, s=1)  
for i in range(len(X_test)):
    Vg, Vd, Id = X_test[i]
    ax.scatter(Vg, Vd, Id, s=1)
_ = ax.set_xlabel('Vg (gate voltage)')
_ = ax.set_ylabel('Vd (drain voltage)')
_ = ax.set_zlabel('Id (drain current)')
_ = ax.set_title('Linear')


fig2 = plt.figure(figsize=(12, 12))
ax2 = fig2.add_subplot(projection='3d')
for i in range(len(X_train_val_log)):
    Vg, Vd, Id = X_train_val_log[i]
    ax2.scatter(Vg, Vd, Id,s=1)  
for i in range(len(X_test_log)):
    Vg, Vd, Id = X_test_log[i]
    ax2.scatter(Vg, Vd, Id, s=1)
_ = ax2.set_xlabel('Vg (gate voltage)')
_ = ax2.set_ylabel('Vd (drain voltage)')
_ = ax2.set_zlabel('Id (drain current)')
_ = ax2.set_title('Log')
'''

"fig = plt.figure(figsize=(12, 12))\nax = fig.add_subplot(projection='3d')\n\nfor i in range(len(X_train_val)):\n    Vg, Vd, Id = X_train_val[i]\n    ax.scatter(Vg, Vd, Id, s=1)  \nfor i in range(len(X_test)):\n    Vg, Vd, Id = X_test[i]\n    ax.scatter(Vg, Vd, Id, s=1)\n_ = ax.set_xlabel('Vg (gate voltage)')\n_ = ax.set_ylabel('Vd (drain voltage)')\n_ = ax.set_zlabel('Id (drain current)')\n_ = ax.set_title('Linear')\n\n\nfig2 = plt.figure(figsize=(12, 12))\nax2 = fig2.add_subplot(projection='3d')\nfor i in range(len(X_train_val_log)):\n    Vg, Vd, Id = X_train_val_log[i]\n    ax2.scatter(Vg, Vd, Id,s=1)  \nfor i in range(len(X_test_log)):\n    Vg, Vd, Id = X_test_log[i]\n    ax2.scatter(Vg, Vd, Id, s=1)\n_ = ax2.set_xlabel('Vg (gate voltage)')\n_ = ax2.set_ylabel('Vd (drain voltage)')\n_ = ax2.set_zlabel('Id (drain current)')\n_ = ax2.set_title('Log')\n"

In [152]:
'''fig.savefig('plots/loadedDataLin.png')
fig2.savefig('plots/loadedDataLog.png')
plt.close()'''

"fig.savefig('plots/loadedDataLin.png')\nfig2.savefig('plots/loadedDataLog.png')\nplt.close()"

### Prepare

In [153]:
# Only grab drain current for now, because we have a fixed set of biases. 
# Combine the linear and log scale data (linear first, then log) into the same target dataset

lin_log_X_train = []
for i in range(0,len(X_train_val)):
    flattened = X_train_val[i][2]
    logId = X_train_val_log[i][2]
    flattened.extend(logId)
    lin_log_X_train.append(np.asarray(flattened))

lin_log_X_test = []
for i in range(0,len(X_test)):
    flattened2 = X_test[i][2]
    logId2 = X_test_log[i][2]
    flattened2.extend(logId2)
    lin_log_X_test.append(np.asarray(flattened2))

print(lin_log_X_test[0].shape)
#print(lin_log_X_test[0])


(370,)


'lin_log_X_train = [np.concatenate([X_train_val[i][2], X_train_val_log[i][2]]) for i in range(len(X_train_val))]\nlin_log_X_test  = [np.concatenate([X_test[i][2], X_test_log[i][2]]) for i in range(len(X_test))]\n\nlin_log_X_train = np.array(lin_log_X_train)\nlin_log_X_test  = np.array(lin_log_X_test)\n\nprint(lin_log_X_test[0].shape)'

## MLP

### Create model

Create a classical multi-layer perceptron for regression: 602 input features, 10 targets, and a certain number of hidden layers and neurons. Taking some inspiration from [Deep learning-based I-V Global Parameter Extraction for BSIM-CMG](https://www.sciencedirect.com/science/article/pii/S003811012300179X), Solid-State Electronics, Vol. 209, November 2023.

Reccomended to download a third party app like "Sleep control Center" or "Amphetamine" to prevent computer from sleeping during the many hour/day long training process

### Create Model by Hand

In [119]:
if KERAS_TUNER == False:
    
    # Multilayer perceptron (MLP) with 601 input features. MLP is having neurons that adjust rules based on how accurate they can guess things
    model_shape = 'mlp_602_'
    
    # Define the number on neurons in the inner layer (in parameter file)
    model_shape += '_'.join(str(l) for l in NEURONS_PER_LAYER)
    
    # 10 output neurons for 10 parameters
    model_shape += '_10'

In [120]:
if KERAS_TUNER == False:
    # Initialize a sequential model, which lets us build a linear stack of layers
    model = Sequential()
    
    # Add an input layer to the model. This layer defines the shape of the input data (602 features per sample), sets dimensions of following layers
    model.add(Input(shape=(370,), name='input1'))
    
    # Iterate over the configuration of neurons for each hidden layer specified in NEURONS_PER_LAYER
    for i, n in enumerate(NEURONS_PER_LAYER):
        # Add a fully connected (dense) hidden layer with spec ified number of neurons
        # The LeCun uniform initializer is used when initializing weights, this makes the model more stable
        # L2 regularization is used in each layer to penalizing large weights, which prevents overfitting
        model.add(Dense(n, name='fc{}'.format(i), kernel_initializer='lecun_uniform', kernel_regularizer=tf.keras.regularizers.l2(0.001)))
    
        # Apply a Leaky ReLU activation function to the outputs of the dense layer
        # This introduces non-linearities, allowing the network to learn complex functions
        # Leaky ReLU is chosen over standard ReLU to help mitigate the "dying ReLU" problem:
        #     - This problem is when neurons using the ReLU activation function output zero for all inputs and stop learning
        #     - Can be mitigated by using variations like Leaky ReLU or proper initialization
        model.add(LeakyReLU(negative_slope=0.01, name='leaky_relu{}'.format(i)))
        
        # Add a dropout layer to reduce overfitting -- randomly drops a set fraction (like 30%) of outputs from the layer
        model.add(Dropout(rate=TRAIN_DROPOUT_RATE, name='dropout{}'.format(i)))
    
    # Add the output layer consisting of 10 neurons, corresponding to the 10 target variables we aim to predict.
    # The same LeCun uniform initializer is used to ensure consistency and stability at the output layer as well.
    model.add(Dense(7, name='fc{}'.format(len(NEURONS_PER_LAYER)), kernel_initializer='lecun_uniform'))
    

In [121]:
if KERAS_TUNER == False:
    # The exponential decay learning rate schedule gradually reduces the learning rate, fine-tuning the learning process for better convergence
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=LR_INITIAL,  
        decay_steps=LR_DECAY_STEPS,        
        decay_rate=LR_DECAY_RATE,          
        staircase=LR_STAIRCASE             
    )
    
    # Set model to minimize loss specified by TRAIN_LOSS, and also to report the loss during training
    model.compile(
        optimizer=tf.optimizers.Adam(learning_rate=lr_schedule),  
        loss=TRAIN_LOSS,                                         
        metrics=[TRAIN_LOSS]                                     
    )

In [122]:
if KERAS_TUNER == False:
    !mkdir -p model
    best_model_file = 'model/{}_best_model.h5'.format(model_shape)
    last_model_file = 'model/{}_last_model.h5'.format(model_shape)

Enable training (`train_and_save`) to overwrite the model file.

In [154]:
train_and_save = True

We use Adam optimizer, minimize the Mean Squared Logarithmic Error, and early stop.

#### Training

In [124]:
from IPython.display import clear_output
if KERAS_TUNER == False:
    class TrainingPlot(tf.keras.callbacks.Callback):
         
        # This function is called when the training begins
        def on_train_begin(self, logs={}):
            # Initialize the lists for holding the logs, losses 
            self.losses = []
            self.val_losses = []
            self.logs = []
        
        # This function is called at the end of each epoch
        def on_epoch_end(self, epoch, logs={}):
            
            # Append the logs, losses  to the lists
            self.logs.append(logs)
            self.losses.append(logs.get('loss'))
            self.val_losses.append(logs.get('val_loss'))
            
            # Before plotting ensure at least 2 epochs have passed
            if len(self.losses) > 1:
                
                # Clear the previous plot
                clear_output(wait=True)
                N = np.arange(0, len(self.losses))
                
                # Plot train loss, train acc, val loss and val acc against epochs passed
                plt.figure()
                plt.plot(N, self.losses, label = "train_loss")
                plt.plot(N, self.val_losses, label = "val_loss")
                plt.title("Training Loss [Epoch {}]".format(epoch))
                plt.xlabel("Epoch #")
                plt.ylabel("Loss/Accuracy")
                plt.legend()
                plt.show()
           
    class LearningRateMonitor(tf.keras.callbacks.Callback):
        def __init__(self):
            super(LearningRateMonitor, self).__init__()
            self.learning_rates = []
    
        def on_epoch_end(self, epoch, logs=None):
            lr = self.model.optimizer._current_learning_rate.read_value()
            self.learning_rates.append(lr)
            print(f"Epoch: {epoch + 1}, Learning Rate: {lr:.10f}")

In [155]:
print(KERAS_TUNER)

True


In [156]:
if KERAS_TUNER == False:
    
    history = None  
    if train_and_save: 
        # Set up early stopping to prevent overfitting by halting training when validation loss stops improving
        early_stopping = EarlyStopping(
            monitor='val_loss',                      # Monitor validation loss for stopping criteria 
            mode='min',                              # Stop when the monitored quantity has stopped decreasing
            patience=TRAIN_EARLY_STOPPING_PATIENCE,  # Number of epochs to wait after last improvement
            verbose=1                                # Enable logging when early stopping happens
        )
    
        # Set up model checkpointing to save the model at its best validation loss:
        model_checkpoint = ModelCheckpoint(
            filepath=best_model_file,          
            monitor='val_loss',            # Save the model based on validation loss improvement
            mode='min',                    # Favor lower validation loss values for saving (minimize)
            save_best_only=True,           # Save only when validation loss improves
            verbose=0                      # No logging for model saving
        )
        
        plot_callback = TrainingPlot()      # Plot training progress
        lr_monitor = LearningRateMonitor()  # Watch learning rate changes
    
        # Train the model on the training data and validate on a portion of it
        history = model.fit(
            np.asarray(lin_log_X_train),  
            np.asarray(y_train_val),      
            epochs=100,                   
            batch_size=TRAIN_BATCH_SIZE,  
            validation_split=TRAIN_VALIDATION_SPLIT,  
            callbacks=[],#[early_stopping, model_checkpoint, plot_callback, lr_monitor],  
            verbose=1  # Enable logging of the training process.
        )
        
        model.save(last_model_file)  # Save the final model when done training!


Load the saved best model and use it from now on.

In [157]:
if KERAS_TUNER == False:
    model = load_model(best_model_file, custom_objects={})

### Keras Tuner to Find Best Hyperparameters and Model

Run this if you want to use keras tuner to make the model rather than doing it by hand

In [158]:
#if using tensorflow
'''from tensorflow.keras import Sequential, Model
from tensorflow.keras.layers import Dense, Input, Dropout, Lambda, LeakyReLU
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from keras_tuner import HyperModel, RandomSearch
from tensorflow.keras.models import load_model'''

'from tensorflow.keras import Sequential, Model\nfrom tensorflow.keras.layers import Dense, Input, Dropout, Lambda, LeakyReLU\nfrom tensorflow.keras.regularizers import l2\nfrom tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint\nfrom keras_tuner import HyperModel, RandomSearch\nfrom tensorflow.keras.models import load_model'

In [217]:
#if using pytorch/keras
import keras
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras import ops
from keras.layers import Dense, Input, Dropout, LeakyReLU, ReLU
from keras.models import Sequential, load_model
from keras.regularizers import l2
from keras_tuner import HyperModel, RandomSearch
from keras.optimizers import Adam
from keras.optimizers.schedules import ExponentialDecay
import keras_tuner
from keras.models import Sequential, load_model, Model

In [161]:
#loading surrogate
surrogate_model = load_model('model_sur/best_keras_model-100k.keras')
surrogate_model.trainable = False
for layer in surrogate_model.layers:
    layer.trainable = False

def surrogate_iv_loss(y_true_iv, y_pred_params):
    # A. Feed predicted parameters into the frozen surrogate model
    raw_reconstructed_iv = surrogate_model(y_pred_params)

    # B. Apply normalized space boundary floor
    FLOOR_VAL_SCALED = 0.0  
    floored_iv = ops.maximum(raw_reconstructed_iv, FLOOR_VAL_SCALED)

    # C. Unscale both ground truth and reconstructed curves to physical Amps
    true_log = y_true_iv * (MAX_LOG - MIN_LOG) + MIN_LOG
    pred_log = floored_iv * (MAX_LOG - MIN_LOG) + MIN_LOG

    true_amps = ops.exp(true_log)
    pred_amps = ops.exp(pred_log)

    true_floored = ops.maximum(true_amps, MIN_LIN)
    pred_floored = ops.maximum(pred_amps, MIN_LIN)

    # D. Compute log-MSE loss
    error = ops.log(pred_floored) - ops.log(true_floored)
    return ops.mean(ops.square(error))

In [198]:
#if using pytorch/keras
if KERAS_TUNER:
    def build_hypermodel_with_limits(hp):
        min_parameters = 12000       # this is a FLOOR, not a cap -- see note below

        layer_num = 4 if FOUR_LAYERS else 3

        neurons_per_layer = [
            hp.Int(f'neurons_{i}', min_value=32, max_value=3000, step=32)
            for i in range(layer_num)
        ]

        input_size, output_size = 602, 7
        total_parameters = input_size * neurons_per_layer[0] + neurons_per_layer[0]
        for i in range(1, layer_num):
            total_parameters += (neurons_per_layer[i-1] * neurons_per_layer[i]
                                 + neurons_per_layer[i])
        total_parameters += neurons_per_layer[-1] * output_size + output_size

        if total_parameters < min_parameters:
            raise keras_tuner.errors.FailedTrialError(
                f"{total_parameters} params below floor of {min_parameters}")

        model = Sequential()
        model.add(Input(shape=(input_size,), name='input1'))

        if TUNER_DROPOUT:
            dropout_rate = hp.Float('dropout_rate', TRAIN_DROPOUT_RATE, 0.5, step=0.1)

        for i, n in enumerate(neurons_per_layer):
            model.add(Dense(n, name=f'fc{i}',
                            kernel_initializer='lecun_uniform',
                            kernel_regularizer=l2(0.001)))
            model.add(LeakyReLU(negative_slope=0.01, name=f'leaky_relu{i}'))
            if TUNER_DROPOUT:
                model.add(Dropout(rate=dropout_rate, name=f'dropout{i}'))

        model.add(Dense(output_size, name='output',
                        kernel_initializer='lecun_uniform'))

        lr_initial = hp.Float('learning_rate', 1e-5, 5e-3, sampling='LOG', default=1e-4)
        #lr_initial = hp.Float('learning_rate', 5e-5, 5e-4, sampling='LOG', default=1e-4)
        lr_schedule = ExponentialDecay(
            initial_learning_rate=lr_initial,
            decay_steps=LR_DECAY_STEPS,
            decay_rate=LR_DECAY_RATE,
            staircase=LR_STAIRCASE,
        )

        model.compile(
            #optimizer=Adam(learning_rate=lr_schedule),
            optimizer=Adam(learning_rate=lr_schedule, clipnorm=1.0),
            loss='mean_squared_error',
            metrics=['mean_squared_error'],
            #metrics=['mean_absolute_error'],
        )
        return model

    

In [199]:
#if using tensor flow
'''if KERAS_TUNER:
    def build_hypermodel(hp):
            layer_num = 4 if FOUR_LAYERS else 3
        
            neurons_per_layer = [
                hp.Int(f'neurons_{i}', min_value=32, max_value=3000, step=32)
                for i in range(layer_num)
            ]
            if TUNER_DROPOUT:
                dropout_rate = hp.Float('dropout_rate', TRAIN_DROPOUT_RATE, 0.15, step=0.05)
                #dropout_rate = hp.Float('dropout_rate', TRAIN_DROPOUT_RATE, 0.5, step=0.1)
        
            param_mins_np = np.zeros(7, dtype=np.float32)
            param_maxs_np = np.ones(7, dtype=np.float32)
        
            iv_input = Input(shape=(370,), name='input1')
            x = iv_input
            for i, n in enumerate(neurons_per_layer):
                x = Dense(n, name=f'fc{i}',
                          kernel_initializer='lecun_uniform',
                          kernel_regularizer=l2(1e-6))(x)
                x = LeakyReLU(negative_slope=0.01, name=f'leaky_relu{i}')(x)
                if TUNER_DROPOUT:
                    x = Dropout(rate=dropout_rate, name=f'dropout{i}')(x)
        
            predicted_params_scaled = Dense(
                7, activation='sigmoid', name='predicted_parameters_scaled',
                kernel_initializer='lecun_uniform')(x)
        
            unscale_layer = Dense(7, activation=None,
                                name='predicted_parameters_unscaled', trainable=False)
            predicted_params_unscaled = unscale_layer(predicted_params_scaled)
            unscale_layer.set_weights([np.diag(param_maxs_np - param_mins_np),
                                       param_mins_np])
        
            # --- REPLACEMENT START: Fix for the ReLU / Truncation Bug ---
            raw_reconstructed_iv = surrogate_model(predicted_params_scaled)
            
            # Use ops.maximum instead of standard ReLU to safely floor without flatlining negative logs
            FLOOR_VAL_SCALED = 0.0  
            reconstructed_iv = ops.maximum(raw_reconstructed_iv, FLOOR_VAL_SCALED)
        
            model = Model(
                inputs=iv_input,
                outputs={
                    'predicted_parameters_unscaled': predicted_params_unscaled,
                    'floored_reconstructed_iv': reconstructed_iv
                },
                name='Inverse_Parameter_Extractor'
            )
        
            def physical_iv_loss(y_true, y_pred):
                min_log = -16.11809565095832
                max_log = -5.066905776264235
                min_lin = 1e-7
                log_floor = ops.log(min_lin)  # precompute once outside ideally
                y_true_log = y_true * (max_log - min_log) + min_log
                y_pred_log = y_pred * (max_log - min_log) + min_log
                true_f = ops.maximum(y_true_log, log_floor)
                pred_f = ops.maximum(y_pred_log, log_floor)
                error = pred_f - true_f
                return ops.mean(ops.square(error))
            # --- REPLACEMENT END ---
    
            #lr_initial = hp.Float('learning_rate', 1e-5, 1e-3, sampling='LOG', default=1e-4)
            lr_initial = hp.Float('learning_rate', 5e-5, 1e-4, sampling='LOG')#, default=1e-4)
          
    
            lr_schedule = ExponentialDecay(
                initial_learning_rate=lr_initial, 
                decay_steps=LR_DECAY_STEPS, 
                decay_rate=LR_DECAY_RATE, 
                staircase=LR_STAIRCASE
            )
        
            model.compile(
                #optimizer=Adam(learning_rate=lr_schedule),
                optimizer=Adam(learning_rate=lr_schedule, clipnorm=1),
                loss={
                    #'predicted_parameters_unscaled': 'mean_squared_error',
                    'floored_reconstructed_iv': physical_iv_loss
                },
                loss_weights={
                    #'predicted_parameters_unscaled': 0.0,
                    'floored_reconstructed_iv': 1.0
                },
                metrics={
                    'floored_reconstructed_iv': physical_iv_loss
                },
            )
            return model'''


"if KERAS_TUNER:\n    def build_hypermodel(hp):\n            layer_num = 4 if FOUR_LAYERS else 3\n        \n            neurons_per_layer = [\n                hp.Int(f'neurons_{i}', min_value=32, max_value=3000, step=32)\n                for i in range(layer_num)\n            ]\n            if TUNER_DROPOUT:\n                dropout_rate = hp.Float('dropout_rate', TRAIN_DROPOUT_RATE, 0.15, step=0.05)\n                #dropout_rate = hp.Float('dropout_rate', TRAIN_DROPOUT_RATE, 0.5, step=0.1)\n        \n            param_mins_np = np.zeros(7, dtype=np.float32)\n            param_maxs_np = np.ones(7, dtype=np.float32)\n        \n            iv_input = Input(shape=(370,), name='input1')\n            x = iv_input\n            for i, n in enumerate(neurons_per_layer):\n                x = Dense(n, name=f'fc{i}',\n                          kernel_initializer='lecun_uniform',\n                          kernel_regularizer=l2(1e-6))(x)\n                x = LeakyReLU(negative_slope=0.01, name=f'

In [200]:
print(len(lin_log_X_train[1]))

370


In [201]:
#if using pytorch/keras
if KERAS_TUNER:

    def build_hypermodel(hp):
        layer_num = 4 if FOUR_LAYERS else 3
        input_size = 370   # ← MUST equal lin_log_X_train.shape[1] AND surrogate output length
    
        neurons_per_layer = [
            hp.Int(f'neurons_{i}', 32, 3000, step=32) for i in range(layer_num)
        ]
        if TUNER_DROPOUT:
            dropout_rate = hp.Float('dropout_rate', TRAIN_DROPOUT_RATE, 0.5, step=0.1)
    
        param_mins_np = np.zeros(7, dtype=np.float32)
        param_maxs_np = np.ones(7, dtype=np.float32)
    
        iv_input = Input(shape=(input_size,), name='input1')
        x = iv_input
        for i, n in enumerate(neurons_per_layer):
            x = Dense(n, name=f'fc{i}', kernel_initializer='lecun_uniform',
                      kernel_regularizer=l2(1e-6))(x)
            x = LeakyReLU(negative_slope=0.01, name=f'leaky_relu{i}')(x)
            if TUNER_DROPOUT:
                x = Dropout(rate=dropout_rate, name=f'dropout{i}')(x)
    
        predicted_params_scaled = Dense(7, activation='sigmoid',
            name='predicted_parameters_scaled', kernel_initializer='lecun_uniform')(x)
    
        unscale_layer = Dense(7, activation=None,
            name='predicted_parameters_unscaled', trainable=False)
        predicted_params_unscaled = unscale_layer(predicted_params_scaled)
        unscale_layer.set_weights([np.diag(param_maxs_np - param_mins_np), param_mins_np])
    
        # ---- SINGLE OUTPUT: the params ----
        model = Model(inputs=iv_input, outputs=predicted_params_unscaled,
                      name='Inverse_Extractor')
    
        def reconstruction_loss(y_true_iv, y_pred_params_unscaled):
            min_log = -16.11809565095832
            max_log = -5.066905776264235
            min_lin = 1e-7
            recon = ops.maximum(surrogate_model(y_pred_params_unscaled), 0.0)
        
            log_floor = ops.log(min_lin)
            t = ops.maximum(y_true_iv * (max_log - min_log) + min_log, log_floor)
            p = ops.maximum(recon      * (max_log - min_log) + min_log, log_floor)
            return ops.mean(ops.square(p - t))
    
        #lr_initial = hp.Float('learning_rate', 5e-5, 5e-4, sampling='LOG', default=1e-4)
        lr_initial = hp.Float('learning_rate', 1e-6, 5e-3, sampling='LOG', default=1e-4)
        lr_schedule = ExponentialDecay(lr_initial, LR_DECAY_STEPS, LR_DECAY_RATE,
                                       staircase=LR_STAIRCASE)
    
        model.compile(optimizer=Adam(learning_rate=lr_schedule, clipnorm=1.0),
                      loss=reconstruction_loss,
                      metrics=[reconstruction_loss])
        return model


In [202]:
print(TUNER_LIMITS)

False


In [203]:
#print("surrogate input min:", X_train_surrogate.min(axis=0))
#print("surrogate input max:", X_train_surrogate.max(axis=0))

In [204]:
from keras_tuner import Objective
if KERAS_TUNER:
    if TUNER_LIMITS:
        tuner = RandomSearch(
        build_hypermodel_with_limits,
        #objective='val_floored_reconstructed_iv_physical_iv_loss',
        objective= 'val_loss',
        max_trials=500,
        executions_per_trial=1,
        #directory='hyper_tuning',
        directory='/scratch/fast/7DayLifetime/oliviagz/hyper_tuning',
        project_name='mlp_tuning',
       overwrite=True,
        )
    else:
        tuner = RandomSearch(
        build_hypermodel,
        #objective='val_floored_reconstructed_iv_physical_iv_loss',
        objective='val_loss',#Objective('val_floored_reconstructed_iv_physical_iv_loss', direction='min'),
        max_trials=500,
        executions_per_trial=1,
        #directory='hyper_tuning',
        directory='/scratch/fast/7DayLifetime/oliviagz/hyper_tuning',
        project_name='mlp_tuning',
        overwrite=True,
    )

In [205]:
if KERAS_TUNER:
    # Setup Callbacks
    early_stopping = EarlyStopping(
        monitor= 'val_loss',#'val_floored_reconstructed_iv_physical_iv_loss',
        mode='min',
        patience=TRAIN_EARLY_STOPPING_PATIENCE,
        verbose=1,
        #restore_best_weights=True
    )

In [206]:
if KERAS_TUNER:
    #dummy_test_params = np.zeros((num_test_samples, 7), dtype=np.float32)
    #dummy_params_train = np.zeros((len(lin_log_X_train), 7), dtype=np.float32)
    y_train_targets = {

    'floored_reconstructed_iv': np.asarray(lin_log_X_train)
}
    tuner.search(
        np.asarray(lin_log_X_train),
        np.asarray(lin_log_X_train),   # y_true_iv — used inside the loss
        epochs=50, batch_size=32,#256,
        validation_split=TRAIN_VALIDATION_SPLIT,
        callbacks=[early_stopping], verbose=1,
    )

    '''tuner.search(
        np.asarray(lin_log_X_train),
        y_train_targets,
        epochs=50,
        batch_size=256,
        validation_split=TRAIN_VALIDATION_SPLIT,
        callbacks=[early_stopping],
        verbose=1,
    )'''


Trial 500 Complete [00h 02m 41s]
val_loss: 13.622488975524902

Best val_loss So Far: 0.00747727183625102
Total elapsed time: 21h 08m 59s


In [220]:
best_model_file ='newest100k_model.keras' #tuner.get_best_models(num_models=1)[0]

# 2. Extract best trial info and hyperparameters
#best_trial = tuner.oracle.get_best_trials(num_trials=1)[0]
#best_hps = best_trial.hyperparameters

#print(f"--- BEST TRIAL METRICS ---")
#print(f"Best Trial ID: {best_trial.trial_id}")
#print(f"Best Validation Loss: {best_trial.score:.6f}")
#print(f"Best Hyperparameters:\n{best_hps.values}\n")

# 3. Get the learning rate used in the best trial
#lr = best_hps.values.get('learning_rate', 0.001)

# 4. Re-compile to clear the Keras Adam optimizer warning
best_model = load_model(
    best_model_file,
    #custom_objects={"physical_iv_loss": physical_iv_loss},
    compile=False,)
best_model.compile(
    #optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
    loss='mse',  # Change to 'huber' if you trained with Huber loss
    metrics=['mae'],
)

# 5. Save the best model as a single file
#save_path = f'{home}/ngspice-skywater-sims/ml_method1/model/best_model.keras'
#best_model.save(save_path)
#print(f"Saved best model successfully to: {save_path}")

In [188]:
'''if KERAS_TUNER:
    model = tuner.get_best_models(num_models=1)[0]
    print(model) '''
# Check your output layer shape
print(best_model.output_shape)
best_model = tuner.get_best_models(num_models=1)[0]

(None, 7)


In [189]:

#test_loss_result = best_model.evaluate(np.asarray(y_test), np.asarray(lin_log_X_test))[0]
#print(y_test[0])
#print(test_loss_result)
#import numpy as np
#from sklearn.metrics import mean_squared_error, mean_absolute_error

# 1. Run predictions through the model


In [221]:
import keras
from keras.models import load_model

# 1. Retrieve the best model and hyperparameters
best_model = tuner.get_best_models(num_models=1)[0]
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best Hyperparameters:")
print(best_hps.values)

# 2. Format test targets as a dictionary matching your multi-output model structure
num_test_samples = len(lin_log_X_test)
dummy_test_params = np.zeros((num_test_samples, 7), dtype=np.float32)

y_test_targets = {
    'predicted_parameters_unscaled': dummy_test_params,
    'floored_reconstructed_iv': np.asarray(lin_log_X_test)
}

# 3. Evaluate the best model on your test set using the target dictionary
eval_results = best_model.evaluate(
    np.asarray(lin_log_X_test), 
    y_test_targets
)
print(f"Verified Best Model Evaluation Results: {eval_results}")

# 4. Save and reload the model
if KERAS_TUNER:
    model_file = 'model/best_keras_model.keras'
    best_model.save(model_file)
    
    # Load the model back safely using keras.models.load_model
    model = load_model(
        model_file, 
        custom_objects={'physical_iv_loss': physical_iv_loss}, 
        safe_mode=False
    )

# 5. Re-compile to build the metrics structure cleanly with both outputs
model.compile(
    optimizer='adam', 
    loss={
        'predicted_parameters_unscaled': 'mean_squared_error',
        'floored_reconstructed_iv': physical_iv_loss
    },
    loss_weights={
        'predicted_parameters_unscaled': 0.0,
        'floored_reconstructed_iv': 1.0
    },
    metrics={
        'floored_reconstructed_iv': physical_iv_loss
    }
)

# 6. MLP visualization using backend-agnostic Keras utils
keras.utils.plot_model(
    model, 
    to_file='plots/best_mlp_architecture.png', 
    show_shapes=True, 
    show_layer_names=True
)

Best Hyperparameters:
{'neurons_0': 96, 'neurons_1': 896, 'neurons_2': 864, 'dropout_rate': 0.0, 'learning_rate': 0.0012238943197319528}


RuntimeError: The size of tensor a (370) must match the size of tensor b (7) at non-singleton dimension 1

### View the model

In [222]:
#if this gives an error do 'conda activate cryo-modelling-env' and 'conda install -c conda-forge python-graphviz'
#tf.keras.utils.plot_model(model, to_file='{}_model.png'.format(model_shape), show_shapes=True, show_layer_names=True)
best_model.summary()
with open('model_summary.txt', 'w') as f:
    best_model.summary(print_fn=lambda x: f.write(x + '\n'))

Model: "Inverse_Extractor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input1 (InputLayer)             │ (None, 370)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc0 (Dense)                     │ (None, 96)             │        35,616 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu0 (LeakyReLU)         │ (None, 96)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout0 (Dropout)              │ (None, 96)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 896)            │        86,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu1 (LeakyReLU)         │ (None, 896)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout1 (Dropout)              │ (None, 896)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 864)            │       775,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu2 (LeakyReLU)         │ (None, 864)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout2 (Dropout)              │ (None, 864)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predicted_parameters_scaled     │ (None, 7)              │         6,055 │
│ (Dense)                         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predicted_parameters_unscaled   │ (None, 7)              │            56 │
│ (Dense)                         │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 903,647 (3.45 MB)

 Trainable params: 903,591 (3.45 MB)

 Non-trainable params: 56 (224.00 B)

### Evaluation

Although we may plot and print many metrics, we focus only on **Mean Squared Error (MSE).**

Plot training history.

In [223]:
#%matplotlib ipympl
%matplotlib inline
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [225]:
import csv
TRAIN_BATCH_SIZE = 256
if 'model' in locals() or 'model' in globals():
    model_shape = "-".join([str(layer.units) for layer in model.layers if hasattr(layer, 'units')])
else:
    model_shape = "Unknown_Shape"
csv_data = [[
    PARAM,
    DATA_AUGMENTATION,
    model_shape,
    #test_loss_result,
    TRAIN_LOSS,
    TRAIN_DROPOUT_RATE,
    TRAIN_EARLY_STOPPING_PATIENCE,
    TRAIN_BATCH_SIZE,
    TRAIN_VALIDATION_SPLIT,
    LR_INITIAL,
    LR_DECAY_STEPS,
    LR_DECAY_RATE,
    LR_STAIRCASE
    ]]

csv_file = 'history_losses.csv'  #this doesnt reqrite this file so you need to delete this if you want something fresh

if not os.path.exists(csv_file):
    with open(csv_file, 'w') as file:
        file.write('data_param,data_augmentation,model_shape,test_loss,train_loss,train_dropout_rate,train_early_stop_patience,train_batch_size,train_val_split,lr_initial,lr_decay_step,lr_decay_rate,lr_stair_case\n')
with open(csv_file, mode='a', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(csv_data)

# Convert data to DataFrame for easier display
df = pd.read_csv(csv_file)

def color_red_column(s):
    return ['color: red' if v else '' for v in s]

styled_df = df.style.apply(color_red_column, subset=['test_loss'])

# Display the DataFrame as a table
display(styled_df)
#qgrid_widget = qgrid.show_grid(df, show_toolbar=True)

,data_param,data_augmentation,model_shape,test_loss,train_loss,train_dropout_rate,train_early_stop_patience,train_batch_size,train_val_split,lr_initial,lr_decay_step,lr_decay_rate,lr_stair_case
0,iv,True,4000-500-4600-7,0.06244721636176109,mean_squared_error,0,50,32.000000,0.200000,0.000100,100.000000,0.99,False
1,iv,True,4000-500-4600-7,0.06244721636176109,mean_squared_error,0,50,32.000000,0.200000,0.000100,100.000000,0.99,False
2,iv,True,4000-500-4600-7,0.06244721636176109,mean_squared_error,0,50,32.000000,0.200000,0.000100,100.000000,0.99,False
3,iv,True,4000-500-4600-7,0.06244721636176109,mean_squared_error,0,50,32.000000,0.200000,0.000100,100.000000,0.99,False
4,iv,True,3200-2700-3400-7,0.06232564523816109,mean_squared_error,0,50,32.000000,0.200000,0.000100,100.000000,0.99,False
5,iv,True,4100-600-2300-7,0.06205158680677414,mean_squared_error,0,50,32.000000,0.200000,0.000100,100.000000,0.99,False
6,iv,True,1300-700-5000-7,0.0617319718003273,mean_squared_error,0,50,32.000000,0.200000,0.000100,100.000000,0.99,False
7,iv,True,3700-600-600-7,0.06258048862218857,mean_squared_error,0,50,32.000000,0.200000,0.000100,100.000000,0.99,False
8,iv,True,1000-1600-4100-7,0.06308142840862274,mean_squared_error,0,50,32.000000,0.200000,0.000100,100.000000,0.99,False
9,iv,True,2100-3000-1100-7,0.06336300075054169,mean_squared_error,0,50,32.000000,0.200000,0.000100,100.000000,0.99,False


In [227]:
TRAIN_BATCH_SIZE = 256
y_pred = best_model.predict(np.array(lin_log_X_test))

34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


### Scaled Test vs Predictions if Y is scaled

In [206]:
if SCALE_Y:
    paramNames = ['vth0_nom', 'vsat_nom', 'rdsw_nom', 'u0_nom', 'nfactor_nom', 'eta0_nom', 'delta_nom']
    
    # Extract the predicted parameters array from the prediction dictionary
    pred_params_array = y_pred['predicted_parameters_unscaled']
    
    pow2_errors = [(y_test[i] - pred_params_array[i])**2 for i in range(len(y_test))]
    abs_errors = [abs(y_test[i] - pred_params_array[i]) for i in range(len(y_test))]
    min_val = 1
    '''for i in range(2): 
        if abs_errors[i].any()<min: 
            min = abs_errors[i]
        print("IV curve #: " + str(i+1))
        for l in range(len(y_test[i])):
            print('Param: {}, Ref: {}, Pred: {}, Pow2Er: {}, AbsEr: {}'.format(paramNames[l],y_test[i][l], y_pred[i][l], pow2_errors[i][l], abs_errors[i][l]))
        print("")'''

### Unscaled Test vs Predictions

In [207]:
if SCALE_Y:
    paramNames = ['vth0_nom', 'vsat_nom', 'rdsw_nom', 'u0_nom', 'nfactor_nom', 'eta0_nom', 'delta_nom']

# 1. Extract the predicted parameters array from the prediction dictionary
pred_params = y_pred['predicted_parameters_unscaled']

if SCALE_Y:
    y_test_unscaled = np.array(y_test, copy=True)
    y_pred_unscaled = np.array(pred_params, copy=True)
    
    # Efficiently inverse-transform column by column using joblib scalers
    for j, name in enumerate(paramNames):
        scaler = joblib.load(f'scalers/scaler_y_{name}.save')
        
        # Unscale test set column
        col_test = y_test_unscaled[:, j].reshape(-1, 1)
        y_test_unscaled[:, j] = scaler.inverse_transform(col_test).ravel()
        
        # Unscale prediction set column
        col_pred = y_pred_unscaled[:, j].reshape(-1, 1)
        y_pred_unscaled[:, j] = scaler.inverse_transform(col_pred).ravel()
    
    print('y_test_unscaled[0]:  ', y_test_unscaled[0])
    print('y_pred_unscaled[0]:  ', y_pred_unscaled[0])
    
    # Vectorized error calculations
    pow2_errors = (y_test_unscaled - y_pred_unscaled) ** 2
    abs_errors = np.abs(y_test_unscaled - y_pred_unscaled)
    
    min_val = 1.0
    for i in range(min(2, len(y_test_unscaled))):
        if np.any(abs_errors[i] < min_val):
            min_val = np.min(abs_errors[i])
        
        print(f"IV curve #: {i + 1}")
        for l in range(len(paramNames)):
            print(f'Param: {paramNames[l]}, Ref: {y_test_unscaled[i][l]:.4f}, '
                  f'Pred: {y_pred_unscaled[i][l]:.4f}, Pow2Er: {pow2_errors[i][l]:.4f}, '
                  f'AbsEr: {abs_errors[i][l]:.4f}')
        print("")

else:
    y_test_arr = np.array(y_test)
    y_pred_arr = np.array(pred_params)
    
    pow2_errors = (y_test_arr - y_pred_arr) ** 2
    abs_errors = np.abs(y_test_arr - y_pred_arr)
    
    min_val = 1.0
    for i in range(min(2, len(y_test_arr))):
        if np.any(abs_errors[i] < min_val):
            min_val = np.min(abs_errors[i])
            
        print(f"IV curve #: {i + 1}")
        for l in range(len(paramNames)):
            print(f'Param: {paramNames[l]}, Ref: {y_test_arr[i][l]:.4f}, '
                  f'Pred: {y_pred_arr[i][l]:.4f}, Pow2Er: {pow2_errors[i][l]:.4f}, '
                  f'AbsEr: {abs_errors[i][l]:.4f}')
        print("")

y_test_unscaled[0]:   [ 0.43628574  9.61699436  3.82541282  5.1997717  20.4347902   2.55316936
  5.42197364]
y_pred_unscaled[0]:   [ 0.3769816  9.898674   3.4027953  3.70544   25.61465    9.454324
  9.888511 ]
IV curve #: 1
Param: vth0_nom, Ref: 0.4363, Pred: 0.3770, Pow2Er: 0.0035, AbsEr: 0.0593
Param: vsat_nom, Ref: 9.6170, Pred: 9.8987, Pow2Er: 0.0793, AbsEr: 0.2817
Param: rdsw_nom, Ref: 3.8254, Pred: 3.4028, Pow2Er: 0.1786, AbsEr: 0.4226
Param: u0_nom, Ref: 5.1998, Pred: 3.7054, Pow2Er: 2.2330, AbsEr: 1.4943
Param: nfactor_nom, Ref: 20.4348, Pred: 25.6147, Pow2Er: 26.8310, AbsEr: 5.1799
Param: eta0_nom, Ref: 2.5532, Pred: 9.4543, Pow2Er: 47.6259, AbsEr: 6.9012
Param: delta_nom, Ref: 5.4220, Pred: 9.8885, Pow2Er: 19.9500, AbsEr: 4.4665

IV curve #: 2
Param: vth0_nom, Ref: 5.4024, Pred: 6.3239, Pow2Er: 0.8490, AbsEr: 0.9214
Param: vsat_nom, Ref: 7.7821, Pred: 0.1000, Pow2Er: 59.0138, AbsEr: 7.6820
Param: rdsw_nom, Ref: 6.1420, Pred: 10.0000, Pow2Er: 14.8843, AbsEr: 3.8580
Param: u0_n

## Compare predictions vs. test set

In [208]:
%matplotlib ipympl
#import matplotlib.pyplot as plt
import matplotlib.pyplot as plt


import matplotlib.cm as cm

In [209]:
'''#abs_errors is now a list of 2d arrays like array([0.01318291, 4.47633068]) with the two values
vth0_abs_errors = [arr[0] for arr in abs_errors]
vsat_abs_errors = [arr[1] for arr in abs_errors]
rdsw_abs_errors = [arr[2] for arr in abs_errors]
u0_abs_errors = [arr[3] for arr in abs_errors]
nfactor_abs_errors = [arr[4] for arr in abs_errors]
eta0_abs_errors = [arr[5] for arr in abs_errors]
delta_abs_errors = [arr[6] for arr in abs_errors]
#cit_abs_errors = [arr[7] for arr in abs_errors]
#prwg_abs_errors = [arr[8] for arr in abs_errors]
#eu_abs_errors = [arr[9] for arr in abs_errors]'''

In [210]:
print("--- DIAGNOSTIC CHECK FOR RDSW ---")
print("len(X_test):", len(X_test))
print("len(rdsw_abs_errors):", len(rdsw_abs_errors))

# Check item shapes
sample_0 = X_test[0]
print("Shape of X_test[0]:", [np.shape(x) for x in sample_0])

# Check data types and special values in X_test
Vg_0 = sample_0[0]
Vd_0 = sample_0[1]
Id_0 = sample_0[2][:185]

print("Vg type & shape:", type(Vg_0), np.shape(Vg_0))
print("Vd type & shape:", type(Vd_0), np.shape(Vd_0))
print("Id type & shape:", type(Id_0), np.shape(Id_0))

# Check for complex numbers or object dtypes
print("Vg dtype:", Vg_0.dtype if hasattr(Vg_0, 'dtype') else "No dtype")
print("Vd dtype:", Vd_0.dtype if hasattr(Vd_0, 'dtype') else "No dtype")
print("Id dtype:", Id_0.dtype if hasattr(Id_0, 'dtype') else "No dtype")

--- DIAGNOSTIC CHECK FOR RDSW ---
len(X_test): 10781
len(rdsw_abs_errors): 10781
Shape of X_test[0]: [(185,), (185,), (370,)]
Vg type & shape: <class 'list'> (185,)
Vd type & shape: <class 'list'> (185,)
Id type & shape: <class 'list'> (185,)
Vg dtype: No dtype
Vd dtype: No dtype
Id dtype: No dtype


In [213]:
print(X_test[0])

[list([0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.11, 0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2, 0.21, 0.22, 0.23, 0.24, 0.25, 0.26, 0.27, 0.28, 0.29, 0.3, 0.31, 0.32, 0.33, 0.34, 0.35, 0.36, 0.37, 0.38, 0.39, 0.4, 0.41, 0.42, 0.43, 0.44, 0.45, 0.46, 0.47, 0.48, 0.49, 0.5, 0.51, 0.52, 0.53, 0.54, 0.55, 0.56, 0.57, 0.58, 0.59, 0.6, 0.61, 0.62, 0.63, 0.64, 0.65, 0.66, 0.67, 0.68, 0.69, 0.7, 0.71, 0.72, 0.73, 0.74, 0.75, 0.76, 0.77, 0.78, 0.79, 0.8, 0.81, 0.82, 0.83, 0.84, 0.85, 0.86, 0.87, 0.88, 0.89, 0.9, 0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99, 1.0, 1.01, 1.02, 1.03, 1.04, 1.05, 1.06, 1.07, 1.08, 1.09, 1.1, 1.11, 1.12, 1.13, 1.14, 1.15, 1.16, 1.17, 1.18, 1.19, 1.2, 1.21, 1.22, 1.23, 1.24, 1.25, 1.26, 1.27, 1.28, 1.29, 1.3, 1.31, 1.32, 1.33, 1.34, 1.35, 1.36, 1.37, 1.38, 1.39, 1.4, 1.41, 1.42, 1.43, 1.44, 1.45, 1.46, 1.47, 1.48, 1.49, 1.5, 1.51, 1.52, 1.53, 1.54, 1.55, 1.56, 1.57, 1.58, 1.59, 1.6, 1.61, 1.62, 1.63, 1.64, 1.65, 1.66, 1.67, 1.68, 1